# Benchmarking and Improving OCR System for Southeast Asian Languages

## Data Collection

In [1]:
from article_pdf import ArticlePDF
from download import download_article_texts, download_article_pdfs, get_articles_by_language

### Wikipedia Articles

- 100 articles
- Collection of the top 20 most viewed Wikipedia articles from 5 categories ([Wikipedia:Popular pages](https://en.wikipedia.org/wiki/Wikipedia:Popular_pages))
- Format: Exact file path

In [2]:
people = ['Donald_Trump', 'Elizabeth_II', 'Barack_Obama', 'Christiano_Ronaldo', 'Michael_Jackson', 'Elon_Musk', 'Lady_Gaga', 'Adolf_Hitler', 'Eminem', 'Lionel_Messi', 'Justin_Bieber', 'Freddie_Mercury', 'Kim_Kardashian', 'Johnny_Depp', 'Steve_Jobs', 'Dwayne_Johnson', 'Michael_Jordan', 'Taylor_Swift', 'Stephen_Hawking', 'Kanye_West']
countries = ['United_States', 'India', 'United_Kingdom', 'Canada', 'Australia', 'China', 'Russia', 'Japan', 'Germany', 'France', 'Singapore', 'Israel', 'Pakistan', 'Philippines', 'Brazil', 'Italy', 'Netherlands', 'New Zealand', 'Ukraine', 'Spain']
cities = ['New_York_City', 'London', 'Hong_Kong', 'Los_Angeles', 'Dubai', 'Washington,_D.C.', 'Paris', 'Chicago', 'Angelsberg', 'Mumbai', 'San_Francisco', 'Rome', 'Monaco', 'Toronto', 'Tokyo', 'Philadelphia', 'Machu_Picchu', 'Jerusalem', 'Amsterdam', 'Boston'] # Excluding Singapore since listed for countries
life = ['Cat', 'Dog', 'Animal', 'Lion', 'Coronavirus', 'Tiger', 'Human', 'Dinosaur', 'Elephant', 'Virus', 'Horse', 'Photosynthesis', 'Evolution', 'Apple', 'Bird', 'Mammal', 'Potato', 'Polar_bear', 'Shark', 'Snake']
structures = ['Taj_Mahal', 'Burj_Khalifa', 'Statue_of_Liberty', 'Great_Wall_of_China', 'Eiffel_Tower', 'Berlin_Wall', 'Stonehenge', 'Mount_Rushmore', 'Colosseum', 'Auschwitz_concentration_camp', 'Great_Pyramid_of_Giza', 'One_World_Trade_Center', 'Empire_State_Building', 'White_House', 'Petra', 'Large_Hadron_Collider', 'Hagia_Sophia', 'Golden_Gate_Bridge', 'Panama_Canal', 'Angkor_Wat'] # Excluding Machu Picchu since listed for cities

english_titles = people + countries + cities + life + structures

english_articles = []
for english_title in english_titles:
    url = f'https://en.wikipedia.org/wiki/{english_title}'
    article = ArticlePDF(english_title, english_title, url, 'en')
    english_articles.append(article)

- To find language codes: https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all
- Missing articles:
    - Thai:
        - Christiano_Ronaldo
        - Angelsberg
    - Vietnamese:
        - Christiano_Ronaldo
        - Angelsberg
    - Bahasa:
        - Christiano_Ronaldo
        - Angelsberg

In [3]:
thai_articles = get_articles_by_language(english_articles, 'th')
vietnamese_articles = get_articles_by_language(english_articles, 'vi')
bahasa_articles = get_articles_by_language(english_articles, 'id')

Issue fetching for Christiano_Ronaldo in th
Issue fetching for Angelsberg in th
Issue fetching for Christiano_Ronaldo in vi
Issue fetching for Angelsberg in vi
Issue fetching for Christiano_Ronaldo in id
Issue fetching for Angelsberg in id


In [4]:
download_article_pdfs(english_articles, 'data/english')

download_article_pdfs(thai_articles, 'data/thai')

download_article_pdfs(vietnamese_articles, 'data/vietnamese')

download_article_pdfs(bahasa_articles, 'data/bahasa')

In [4]:
# download_article_texts(english_articles, 'data/english')
download_article_texts(thai_articles, 'data/thai')
download_article_texts(vietnamese_articles, 'data/vietnamese')
download_article_texts(bahasa_articles, 'data/bahasa')

# TODO: Get ground truth in Thai, Vietnamese, Bahasa

## Benchmarking

### EasyOCR

In [4]:
import easyocr
reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory

In [9]:
cat_image_1 = reader.readtext('data/english/cat/images/1.png', detail=0)
' '.join(cat_image_1)

"WIKIPEDIA The Free Encyclopedia Cat The cat (Felis catus), commonly referred to as the Cat domestic cat or house cat, is a small domesticated carnivorous mammal. It is the only domesticated species Temporal range: 9,500 years ago - of the family Felidae: Recent advances in archaeology present and genetics have shown that the domestication of the cat occurred in the Near East around 7500 BC. It is commonly kept as a house and farm cat, but also ranges   freely as a feral cat avoiding human contact: Valued by humans for companionship and its ability to kill  vermin, the cat's  retractable claws are adapted to small prey like mice and rats: It has a strong, flexible body, reflexes, and sharp teeth, and its night vision and sense of smell are well developed: It is a social species, but a solitary hunter and a crepuscular predator. Cat communication includes vocalizations like meowing, purring, trilling, hissing, growling, and grunting as well as cat body language. It can hear sounds too f

### Tesseract

In [11]:
import pytesseract

In [13]:
CAT_IMAGES_PATH = 'data/english/cat/images'

s = pytesseract.image_to_string(f'{CAT_IMAGES_PATH}/1.png')
s

'“e°) WIKIPEDIA\n\n‘37 The Free Encyclopedia\n\nCat\n\nThe cat (Felis catus), commonly referred to as the\ndomestic cat or house cat, is a small domesticated\ncarnivorous mammal. It is the only domesticated species\nof the family Felidae. Recent advances in archaeology\nand genetics have shown that the domestication of the\ncat occurred in the Near East around 7500 BC. It is\ncommonly kept as a house pet and farm cat, but also\nranges freely as a feral cat avoiding human contact.\nValued by humans for companionship and its ability to\nkill vermin, the cat\'s retractable claws are adapted to\nkilling small prey like mice and rats. It has a strong,\nflexible body, quick reflexes, and sharp teeth, and its\nnight vision and sense of smell are well developed. It is a\nsocial species, but a solitary hunter and a crepuscular\npredator. Cat communication includes vocalizations like\nmeowing, purring, trilling, hissing, growling, and\ngrunting as well as cat body language. It can hear sounds\nt

In [5]:
from torchmetrics.text import CharErrorRate
cer = CharErrorRate()

predictions = ["this is the prediction", "there is an other sample"]
targets = ["this is the reference", "there is another one"]
print(cer(predictions, targets))

# 0.0 CER
predictions = ["100% match", "this is some text"]
targets = ["100% match", "this is some text"]
print(cer(predictions, targets))

tensor(0.3415)
tensor(0.)
